# 03 Silver: one clean table of readings

Reads the Bronze Parquet tables (everything still text) and builds **one clean table** for both vendors:
`data/silver/readings.parquet`.

One problem per step, in a fixed order, and every step starts by **looking** at the data before fixing it.
Problems that can't be fixed safely are **flagged** in a True/False column, never deleted.

| Step | What it does |
| --- | --- |
| 0 | Setup: load Bronze |
| 1 | Same column names for both vendors |
| 2 | Real types: numbers, then dates in UTC |
| 3 | Clean device ids |
| 4 | Remove duplicates (latest file wins) |
| 5 | Flag impossible values |
| 6 | One table with a `vendor` column |
| 7 | One common state: ON / OFF / ERROR |
| 8 | Save |

⚠️ Cells change `fs` and `so` in place, so run the notebook **from the top with Run All**, not cell by cell twice.

## 0. Setup

Find the project folder and load the two Bronze reading tables (all columns are text).

In [1]:
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
import sys
sys.path.insert(0, str(ROOT / "python"))
from src.silver_rules import to_number, clean_device_id, drop_resent_duplicates, flag_impossible_values
BRONZE = ROOT / "data" / "bronze"

fs = pd.read_parquet(BRONZE / "fusionsolar.parquet")
so = pd.read_parquet(BRONZE / "solis.parquet")
print(fs.shape, so.shape)

(725604, 10) (509342, 9)


## 1. Same column names

One dictionary per vendor maps its names to the Silver names. Units go into the names (`_kw`, `_kwh`, `_c`, `_pct`),
and the device column takes the name of the key in `devices`, so later joins are natural.
Only FusionSolar has `efficiency_pct`.

In [2]:
FS_NAMES = {"device_id": "device_hash_id", "collect_time": "ts",
            "active_power": "ac_power_kw", "dc_power": "dc_power_kw",
            "efficiency": "efficiency_pct", "day_cap": "energy_today_kwh",
            "total_cap": "energy_total_kwh", "temperature": "temperature_c",
            "inverter_state": "state_code"}

SO_NAMES = {"inverter_id": "device_hash_id", "timestamp": "ts",
            "total_ac_power": "ac_power_kw", "dc_power": "dc_power_kw",
            "eToday": "energy_today_kwh", "eTotal": "energy_total_kwh",
            "temperature": "temperature_c", "state": "state_code"}

fs = fs.rename(columns=FS_NAMES)
so = so.rename(columns=SO_NAMES)
print(fs.columns.tolist())
print(so.columns.tolist())

['device_hash_id', 'ts', 'ac_power_kw', 'dc_power_kw', 'efficiency_pct', 'energy_today_kwh', 'energy_total_kwh', 'temperature_c', 'state_code', 'source_file']
['device_hash_id', 'ts', 'ac_power_kw', 'dc_power_kw', 'energy_today_kwh', 'energy_total_kwh', 'temperature_c', 'state_code', 'source_file']


## 2. Real types

### 2a. Numbers

**Look first:** try the conversion with `errors="coerce"` on a copy and list the values that fail.
In Solis about 24,900 values per column fail: decimal commas (`0,08`, from the 17 files with `;`) and the markers `NULL` / `N/A`.

In [3]:
NUM_COLS = ["ac_power_kw", "dc_power_kw", "energy_today_kwh",
            "energy_total_kwh", "temperature_c"]

for col in NUM_COLS:
    as_number = pd.to_numeric(so[col], errors="coerce")
    bad = so[col][as_number.isna()]
    print(col, len(bad))
    print(bad.value_counts().head(), "\n")

ac_power_kw 24914
ac_power_kw
0,0     2678
0,08    1409
0,04    1064
0,02     818
NULL     193
Name: count, dtype: int64 



dc_power_kw 24907
dc_power_kw
0,0       2914
NULL       205
N/A        104
0,0161      56
0,0162      42
Name: count, dtype: int64 



energy_today_kwh 24883
energy_today_kwh
0,0     1424
0,1      591
0,2      450
0,3      273
NULL     172
Name: count, dtype: int64 



energy_total_kwh 24869
energy_total_kwh
76445,0    182
77189,0    182
76267,0    182
75894,0    182
76575,0    181
Name: count, dtype: int64 



temperature_c 24894
temperature_c
NULL    200
36,0    179
34,4    161
40,7    160
34,0    155
Name: count, dtype: int64 



**Fix:** `to_number` changes `,` to `.`, turns `""`, `NULL`, `N/A` into real missing values, and converts
**without** `coerce`, so anything unexpected stops the notebook instead of silently becoming missing.

Result in Solis: unreadable values drop from ~24,900 to ~300 per column (1,461 in total, all sent as missing by the vendor).

In [4]:
for col in NUM_COLS:
    so[col] = to_number(so[col])

print(so[NUM_COLS].dtypes)
print(so[NUM_COLS].isna().sum())

ac_power_kw         float64
dc_power_kw         float64
energy_today_kwh    float64
energy_total_kwh    float64
temperature_c       float64
dtype: object
ac_power_kw         306
dc_power_kw         309
energy_today_kwh    280
energy_total_kwh    277
temperature_c       289
dtype: int64


The same function for FusionSolar. It uses dots already, so the missing counts don't change (2,083 in total).

`state_code` stays text on purpose: it is a label, not an amount.

In [5]:
FS_NUM_COLS = NUM_COLS + ["efficiency_pct"]

for col in FS_NUM_COLS:
    fs[col] = to_number(fs[col])

print(fs[FS_NUM_COLS].dtypes)
print(fs[FS_NUM_COLS].isna().sum())

ac_power_kw         float64
dc_power_kw         float64
energy_today_kwh    float64
energy_total_kwh    float64
temperature_c       float64
efficiency_pct      float64
dtype: object
ac_power_kw         354
dc_power_kw         363
energy_today_kwh    322
energy_total_kwh    345
temperature_c       355
efficiency_pct      344
dtype: int64


### 2b. Dates and times, all in UTC

**Look first.** The two vendors write time very differently:

- FusionSolar: 13-digit numbers, milliseconds since 1 January 1970, already UTC.
- Solis: text in local time (UTC+01:00), usually `2024-06-27 17:05:00`, but some files write the day first: `21/09/2024 23:08:26`.

In [6]:
print(fs["ts"].head(3))
print(so["ts"].head(3))

0    1735686300000
1    1735686000000
2    1735686000000
Name: ts, dtype: str
0    2024-06-27 17:05:00
1    2024-06-27 18:25:00
2    2024-06-27 17:20:00
Name: ts, dtype: str


How many Solis rows write the day first, and in how many files? Expected: 23,877 rows in 17 files.

In [7]:
has_slash = so["ts"].str.contains("/")
print(has_slash.sum())
print(so.loc[has_slash, "source_file"].nunique())

23877
17


**FusionSolar:** text → number → date, reading the number as milliseconds in UTC.

In [8]:
fs["ts"] = pd.to_numeric(fs["ts"])
fs["ts"] = pd.to_datetime(fs["ts"], unit="ms", utc=True)
print(fs["ts"].head(3))
print(fs["ts"].dtype)

0   2024-12-31 23:05:00+00:00
1   2024-12-31 23:00:00+00:00
2   2024-12-31 23:00:00+00:00
Name: ts, dtype: datetime64[us, UTC]
datetime64[us, UTC]


**Solis, step 1:** read the column once per style and combine with `fillna`.
The missing count must be **0**; otherwise a row is in a third style we don't know about.

In [9]:
normal = pd.to_datetime(so["ts"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
dayfirst = pd.to_datetime(so["ts"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
so["ts"] = normal.fillna(dayfirst)
print(so["ts"].isna().sum())
print(so["ts"].head(3))

0
0   2024-06-27 17:05:00
1   2024-06-27 18:25:00
2   2024-06-27 17:20:00
Name: ts, dtype: datetime64[us]


**Solis, step 2:** local time is one hour ahead of UTC, so subtract one hour and label the result UTC.

⚠️ Each run of this cell subtracts another hour. That's why the notebook should be run with Run All.

In [10]:
so["ts"] = so["ts"] - pd.Timedelta(hours=1)
so["ts"] = so["ts"].dt.tz_localize("UTC")
print(so["ts"].head(3))
print(so["ts"].dtype, fs["ts"].dtype)

0   2024-06-27 16:05:00+00:00
1   2024-06-27 17:25:00+00:00
2   2024-06-27 16:20:00+00:00
Name: ts, dtype: datetime64[us, UTC]
datetime64[us, UTC] datetime64[us, UTC]


## 3. Clean device ids

**Look first:** which ids in the readings are not in `devices`? Three problems show up:

1. spaces around some FusionSolar ids,
2. capital letters in some Solis ids,
3. a made-up id, `0000deadbeef...`, in 276 FusionSolar rows from 2 files.

In [11]:
devices = pd.read_parquet(BRONZE / "devices.parquet")
known = devices["device_hash_id"]

fs_bad = fs.loc[~fs["device_hash_id"].isin(known), "device_hash_id"]
so_bad = so.loc[~so["device_hash_id"].isin(known), "device_hash_id"]
print(len(fs_bad), len(so_bad))
print(fs_bad.value_counts().head(5))
print(so_bad.value_counts().head(5))

1663 974
device_hash_id
0000deadbeef0000deadbeef00000000       276
  94aedd81bad035cc73968fe5b06f808a      77
  6187ec08ac3e790c94c9ae81b0af5e1c      51
  dc8fae88167f73e17a152555e7580c1b      48
  22a8a8cda7e139d01f0b204f4a7d80e8      47
Name: count, dtype: int64
device_hash_id
29F08F60676EA3C4E2085071C1957D8C    124
2E1392D8173E175E4FA40A27AA555E09    101
A2BC58707E3AC5B41A4A8F49900D675B     90
372F9D827236B14028B1639E7548A7D2     86
594DF11D8E82584B2EE6278A7CAC9471     76
Name: count, dtype: int64


**Fix:** `strip()` and `lower()` fix problems 1 and 2. Problem 3 can't be fixed (we can't know the real device),
so those rows are flagged with `unknown_device`. Expected: 276 and 0.

In [12]:
fs["device_hash_id"] = clean_device_id(fs["device_hash_id"])
so["device_hash_id"] = clean_device_id(so["device_hash_id"])

fs["unknown_device"] = ~fs["device_hash_id"].isin(known)
so["unknown_device"] = ~so["device_hash_id"].isin(known)
print(fs["unknown_device"].sum(), so["unknown_device"].sum())

276 0


## 4. Remove duplicates

A duplicate is the same device at the same time more than once, usually because the next day's file sent it again.
This only works now that times are real UTC dates and ids are clean (steps 2 and 3).

**Look first:** 31,729 in FusionSolar, 23,002 in Solis.

In [13]:
print(fs.duplicated(subset=["device_hash_id", "ts"]).sum())
print(so.duplicated(subset=["device_hash_id", "ts"]).sum())

31729
23002


**Fix: the latest file wins.** File names contain the date, so sorting by `source_file` puts them in date order,
and `keep="last"` keeps the newest copy, which is usually the corrected one.
Result: 693,875 and 486,340 rows.

In [14]:
fs = drop_resent_duplicates(fs)
so = drop_resent_duplicates(so)
print(len(fs), len(so))

693875 486340


## 5. Flag impossible values

Values we can't safely correct get a True/False column instead of being changed or deleted.

| Flag | Rule | Why |
| --- | --- | --- |
| `bad_temperature` | above 100 °C | sensor error, e.g. 6553.5 |
| `bad_counter` | lifetime energy = 0 | that counter only goes up |
| `unknown_state` | code not in the vendor's mapping table | meaning unknown |
| `too_much_power` | more than 2 × the inverter's size | power sent in W instead of kW |

The temperature rule describes what is impossible, not only the value we injected: in Solis it also catches 29 bad readings
that were already in the original dataset (e.g. 3276.7).

In [15]:
fs_states = pd.read_parquet(BRONZE / "inverter_state_mapping_fusionsolar.parquet")
so_states = pd.read_parquet(BRONZE / "inverter_state_mapping_solis.parquet")
fs["unknown_state"] = ~fs["state_code"].isin(fs_states["state_id"])
so["unknown_state"] = ~so["state_code"].isin(so_states["state_id"])

for col in ["unknown_state"]:
    print(col, fs[col].sum(), so[col].sum())

unknown_state 335 226


### Watts instead of kilowatts

`merge` works like VLOOKUP: it brings each device's `max_power` (kW) next to every reading.
An inverter can't produce more than its size, so readings above 2 × `max_power` were sent in watts.
Expected: 1,249 FusionSolar rows from 10 files, and 13 Solis rows that come from the original data.

In [16]:
devices["max_power"] = pd.to_numeric(devices["max_power"])
size = devices[["device_hash_id", "max_power"]]

fs = fs.merge(size, on="device_hash_id", how="left")
so = so.merge(size, on="device_hash_id", how="left")

fs = flag_impossible_values(fs)
so = flag_impossible_values(so)

for col in ["bad_temperature", "bad_counter", "too_much_power", "bad_energy_today"]:
    print(col, fs[col].sum(), so[col].sum())
print(fs.loc[fs["too_much_power"], "source_file"].nunique())

bad_temperature 136 120
bad_counter 199 139
too_much_power 1249 13
bad_energy_today 1 19
10


## 6. One table

Add a `vendor` column, stack both vendors, and drop `max_power` (it belongs to the device, not the reading).
Solis rows have no `efficiency_pct`. Expected shape: (1180215, 16).

In [17]:
fs["vendor"] = "fusionsolar"
so["vendor"] = "solis"

silver = pd.concat([fs, so], ignore_index=True)
silver = silver.drop(columns=["max_power"])
print(silver.shape)
print(silver.dtypes)



(1180215, 17)
device_hash_id                      str
ts                  datetime64[us, UTC]
ac_power_kw                     float64
dc_power_kw                     float64
efficiency_pct                  float64
energy_today_kwh                float64
energy_total_kwh                float64
temperature_c                   float64
state_code                          str
source_file                         str
unknown_device                     bool
unknown_state                      bool
bad_temperature                    bool
bad_counter                        bool
too_much_power                     bool
bad_energy_today                   bool
vendor                              str
dtype: object


## 7. One common state

Each vendor has its own state codes (`512` for FusionSolar, `1` for Solis). The two mapping tables are stacked with a
`vendor` column, then one merge on **vendor + code** gives every row its canonical state, and `.map` turns 1/2/3 into words.

Check: the row count must stay 1,180,215. If a merge adds rows, the lookup table has duplicates.
Rows with no state are exactly the `unknown_state` rows (335 + 226).

Note for Gold: **Solis is never OFF**, not even at night. For Solis, "online" means connected, not producing.

In [18]:
fs_states["vendor"] = "fusionsolar"
so_states["vendor"] = "solis"
states = pd.concat([fs_states, so_states])
states = states[["vendor", "state_id", "canonical_state_id"]]
states = states.rename(columns={"state_id": "state_code"})

silver = silver.merge(states, on=["vendor", "state_code"], how="left")
silver["state"] = silver["canonical_state_id"].map({"1": "ON", "2": "OFF", "3": "ERROR"})

print(len(silver))
print(silver.groupby("vendor")["state"].value_counts(dropna=False))

1180215
vendor       state
fusionsolar  ON       601478
             OFF       89012
             ERROR      3050
             NaN         335
solis        ON       486114
             NaN         226
Name: count, dtype: int64


## 8. Save

Write the Silver table as Parquet. This is the input for Gold.

In [19]:
SILVER = ROOT / "data" / "silver"
SILVER.mkdir(exist_ok=True)
silver.to_parquet(SILVER / "readings.parquet", index=False)

## Result

`data/silver/readings.parquet`: **1,180,215 readings**, both vendors, with the same columns, real types, all times in UTC,
clean device ids, no duplicates, five flag columns and one common state.

Checked against the answer key: every injected fault was fixed, flagged, or replaced by a clean copy from the next day's file.
For example, 9 of the 236 temperature overflows were overwritten by a clean resend in step 4, and the other 227 are flagged.

Gaps (readings that never arrived) can't be fixed, only found. That's the next step, with a small Silver report in `reports/`.